# Cease & Desist Document Processing - RAG + Few-shot Setup

## Objective
This notebook prepares:
- Document ingestion (PDF + Image OCR)
- RAG (Retrieval-Augmented Generation) pipeline
- Few-shot examples for classification

## Output
- Persistent vector database (Chroma)
- Curated few-shot examples
- Retrieval function for inference

This will be used later in the classification pipeline.

In [2]:
!pip install -U langchain langchain-community langchain-core pymupdf chromadb pypdf tiktoken pytesseract pillow pdf2image poppler-utils

## API Key Configuration

We securely load API keys using Google Colab's `userdata`.

### Keys Used:
- **Mistral API Key** → for advanced OCR (Pixtral model)
- **Groq API Key** → for fast LLM-based classification

---

### Why this step?
- Avoids hardcoding sensitive credentials
- Enables access to external AI services

In [3]:
from google.colab import userdata
import os

os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

## Library Imports & Environment Setup

This cell imports all required libraries for:

### Image Processing
- OpenCV (`cv2`)
- NumPy
- PIL

### OCR
- Tesseract

### LLM & APIs
- Requests
- Base64 encoding

### RAG Pipeline
- LangChain
- ChromaDB
- HuggingFace embeddings

---

### Data Source
We define the path to our dataset stored in Google Drive.

In [4]:
import os
import random
import cv2
import pytesseract
import numpy as np
import base64
from PIL import Image
import requests

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from pdf2image import convert_from_path


DATA_PATH = "drive/MyDrive/data"

## Image Preprocessing

This function improves OCR accuracy by:

### Steps:
1. Rotating image (0°, 90°, 180°, 270°)
2. Selecting the best orientation based on text length
3. Converting to grayscale
4. Applying thresholding

---

### Goal:
Ensure text is clear and readable before OCR extraction

In [5]:
def preprocess_image(path):
    image = cv2.imread(path)

    rotations = [0, 90, 180, 270]
    best_img = image
    best_score = 0

    for angle in rotations:
        rotated = np.rot90(image, k=angle // 90)
        text = pytesseract.image_to_string(rotated)

        score = len(text.strip())
        if score > best_score:
            best_score = score
            best_img = rotated

    gray = cv2.cvtColor(best_img, cv2.COLOR_BGR2GRAY)
    gray = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)[1]

    return gray

## OCR Quality Validation

This function detects poor OCR results.

### Conditions for "Bad OCR":
- Too little text
- Very few words
- High ratio of special characters

---

### Why?
If OCR fails, we switch to a more powerful model (Pixtral)

In [6]:
def is_bad_ocr(text):
    if len(text.strip()) < 50:
        return True

    words = text.split()
    if len(words) < 5:
        return True

    weird_ratio = sum(1 for c in text if not c.isalnum() and c != " ") / len(text)

    return weird_ratio > 0.5

## Image Encoding (Base64)

We convert images into Base64 format.

### Why?
- Required for sending images to APIs (like Mistral)
- Enables embedding image data inside requests

## Advanced OCR using Mistral (Pixtral)

This function uses **Pixtral multimodal model** to:

- Extract text from images
- Fix orientation
- Preserve tables

---

### Retry Logic:
Handles rate limits by retrying API calls

---

### When used?
Only when normal OCR fails (fallback mechanism)

In [8]:
import base64
import requests
import time

def encode_image_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def extract_with_pixtral(image_path):
    url = "https://api.mistral.ai/v1/chat/completions"

    base64_image = encode_image_base64(image_path)

    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "pixtral-12b-2409",
        "messages": [
            {
                "role": "user",
                "content": f"""
Extract all readable text from this document.
Fix orientation.
Preserve table structure if present.
"""
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": f"data:image/jpeg;base64,{base64_image}"
                    }
                ]
            }
        ],
        "max_tokens": 2000
    }

    for attempt in range(3):
        response = requests.post(url, headers=headers, json=payload)
        data = response.json()

        if "choices" in data:
            return data["choices"][0]["message"]["content"]

        if data.get("code") == "1300":
            print("Rate limited, retrying...")
            time.sleep(5)
            continue

        print("Mistral Error:", data)
        return ""

    return ""

## PDF → Image Conversion

Converts PDF pages into images.

### Why?
- OCR works better on images
- Enables uniform processing pipeline

---

### Optimization:
Only first 2 pages are processed to save time

In [9]:
def pdf_to_images(path):
    images = convert_from_path(path, first_page=1, last_page=2)
    return images

## Text Extraction from Images

Pipeline:
1. Preprocess image
2. Run Tesseract OCR
3. Validate OCR quality
4. If bad → use Mistral Pixtral

---

### Smart Decision:
Automatically switches to better OCR when needed

In [10]:
def extract_text_from_image(path):
    processed = preprocess_image(path)
    text = pytesseract.image_to_string(processed)

    if is_bad_ocr(text):
        print(f"Using Mistral for image: {path}")
        return extract_with_pixtral(path)

    return text

In [11]:
from PIL import Image
import fitz

def pdf_to_image(path):
    doc = fitz.open(path)
    page = doc[0]
    pix = page.get_pixmap()
    img_path = "temp.jpg"
    pix.save(img_path)
    return img_path

In [12]:
def extract_text_from_pdf(path):
    try:
        from pypdf import PdfReader

        reader = PdfReader(path)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text

        if len(text.strip()) < 50:
            print(f"Using Mistral for PDF: {path}")
            img_path = pdf_to_image(path)
            return extract_with_pixtral(img_path)

        return text

    except Exception as e:
        print(f"PyPDF failed → using Mistral: {path}")
        img_path = pdf_to_image(path)
        return extract_with_pixtral(img_path)

## Document Loader

This function processes all files and converts them into LangChain `Document` objects.

### Supported Formats:
- PDF
- Images (JPG, PNG, JPEG)

---

### Output:
Each document contains:
- Clean text
- Metadata (filename, type, source)

---

### Skips:
- Empty or unreadable documents

In [13]:
def load_files(file_list):
    docs = []

    for file in file_list:
        path = os.path.join(DATA_PATH, file)

        try:
            if file.lower().endswith(".pdf"):
                text = extract_text_from_pdf(path)
                doc_type = "pdf"

            elif file.lower().endswith((".jpg", ".png", ".jpeg")):
                text = extract_text_from_image(path)
                doc_type = "image"

            else:
                continue

            text = text or ""

            if len(text.strip()) < 10:
                print(f"Skipping empty doc: {file}")
                continue

            docs.append(
                Document(
                    page_content=" ".join(text.split()),  # now safe
                    metadata={
                        "source": path,
                        "filename": file,
                        "doc_type": doc_type
                    }
                )
            )

        except Exception as e:
            print(f" Failed: {file} → {e}")

    return docs

## Dataset Splitting

We randomly divide files into:

- Few-shot set → for examples
- Test set → for evaluation
- RAG set → full dataset

---

### Purpose:
- Train prompts (few-shot)
- Evaluate model (test)
- Provide context (RAG)

In [14]:
files = os.listdir(DATA_PATH)
random.shuffle(files)

few_shot_files = files[:10]
test_files = files[10:20]
rag_files = files

print("Few-shot:", few_shot_files)
print("Test:", test_files)

Few-shot: ['01_copyright_infringement_photography.pdf', '10_breach_of_contract_nda.pdf', '20251009_105307.jpg', 'LOA3.pdf', 'notice_4.pdf', 'bw_doc_5.pdf', '06_harassment_workplace.pdf', 'LOA5.pdf', '20251009_111026.jpg', '04_defamation_online_review.pdf']
Test: ['05_patent_infringement_medical_device.pdf', 'LOA2.pdf', 'LOA9.pdf', 'notice_2.pdf', 'notice_1.pdf', '20251009_110821.jpg', 'notice_3.pdf', 'LoA1.pdf', '20251009_111416.jpg', '20251009_110503.jpg']


## Loading Documents

We load:

- Few-shot documents
- RAG documents
- Test documents

---

### Output:
Structured LangChain documents ready for processing

In [15]:
few_shot_docs = load_files(few_shot_files)
rag_docs = load_files(rag_files)
test_docs = load_files(test_files)

print("Loaded RAG docs:", len(rag_docs))

Using Mistral for PDF: drive/MyDrive/data/notice_4.pdf
Using Mistral for PDF: drive/MyDrive/data/bw_doc_5.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_4.pdf
Using Mistral for PDF: drive/MyDrive/data/bw_doc_5.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_2.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_1.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_3.pdf
Using Mistral for PDF: drive/MyDrive/data/bw_doc_2.pdf
Using Mistral for PDF: drive/MyDrive/data/bw_doc_1.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_5.pdf
Using Mistral for PDF: drive/MyDrive/data/bw_doc_3.pdf
Using Mistral for PDF: drive/MyDrive/data/bw_doc_4.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_2.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_1.pdf
Using Mistral for PDF: drive/MyDrive/data/notice_3.pdf
Loaded RAG docs: 40


## Document Classification (LLM)

We use Groq's LLM to classify documents into:

- Cease
- Uncertain
- Irrelevant

---

### How it works:
- Sends document text to LLM
- Uses prompt-based classification
- Returns only label

---

### Advantage:
Fast inference using Groq API

In [16]:
def classify_with_groq(text):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = f"""
    Classify the document into:
    - cease
    - uncertain
    - irrelevant

    Text:
    {text[:1000]}

    Only return label.
    """

    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [{"role": "user", "content": prompt}]
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        data = response.json()

        if "choices" not in data:
            print("Groq Error:", data)
            return "uncertain"

        return data["choices"][0]["message"]["content"].strip().lower()

    except Exception as e:
        print("Groq Exception:", e)
        return "uncertain"

## Few-Shot Example Generation

We generate labeled examples from documents.

### Process:
1. Take sample documents
2. Classify using LLM
3. Store (text + label)

---

### Purpose:
Improve model performance using examples

In [17]:
def build_few_shots(docs, n=5):
    examples = []

    for d in docs[:n]:
        if not d.page_content:
            continue

        label = classify_with_groq(d.page_content)

        examples.append({
            "text": d.page_content[:500],
            "label": label,
            "file_name": d.metadata.get("filename", "unknown")
        })

    return examples

## Saving Few-Shot Data

We store few-shot examples in a JSON file.

### Why?
- Reusable across runs
- Easy to load into prompts
- Helps in reproducibility

In [18]:
few_shots = build_few_shots(few_shot_docs)

import json
with open("few_shots.json", "w") as f:
    json.dump(few_shots, f, indent=2)

## Document Chunking

We split documents into smaller chunks.

### Parameters:
- Chunk size: 800
- Overlap: 100

---

### Why?
- Better embedding quality
- Improved retrieval performance

## Text Embeddings

We convert text into vector representations using:

**Model:** all-MiniLM-L6-v2

---

### Purpose:
- Enable similarity search
- Power RAG pipeline

## Vector Database (ChromaDB)

We store embeddings in ChromaDB.

### Features:
- Persistent storage
- Fast retrieval
- Scalable

---

### Output:
A retriever object for querying similar documents

In [19]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(rag_docs)

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

/tmp/ipykernel_6537/3963839796.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Context Retrieval (RAG)

This function retrieves similar documents based on a query.

### Steps:
1. Search vector DB
2. Fetch top-k relevant chunks
3. Combine into context

---

### Goal:
Provide LLM with relevant past knowledge

In [20]:
def get_relevant_context(query):
    docs = retriever.get_relevant_documents(query)

    context = "\n\n".join([
        f"Example {i+1}:\n{d.page_content}"
        for i, d in enumerate(docs)
    ])

    return context, docs